Task 3
# Chatbot using Hugging Face Transformers

Step 1: Install required libraries

In [1]:
!pip install transformers torch -q

Step 2: Import libraries

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Load Pre-trained Model
model_name = "microsoft/DialoGPT-medium"

# Load tokenizer (converts text → numbers)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load model (brain of chatbot)
model = AutoModelForCausalLM.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/863M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/863M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [3]:
# This function handles common questions with fixed correct answers
def simple_knowledge_response(text):
   # Convert input to lowercase for easy matching
    text = text.lower()

    # Greetings
    if "hello" in text or "hi" in text:
        return "Hello Sneha! How can I assist you today?"

     # Personal / Basic
    elif "how are you" in text:
        return "I'm doing great! How can I help you today?"

    elif "what is your name" in text:
        return "I am your AI assistant chatbot."

    elif "thank you" in text:
        return "You're welcome! Feel free to ask more questions."

     # Programming
    elif "who created python" in text:
        return "Python was created by by Guido van Rossum in 1991."

    elif "what is python" in text:
        return "Python is a high-level, easy-to-learn programming language used for web development, AI, and data science."

    elif "what is java" in text:
        return "Java is a popular object-oriented programming language used for building applications."


    # AI & ML
    elif "what is ai" in text:
        return "Artificial Intelligence is the ability of machines to perform tasks that require human intelligence."

    elif "what is machine learning" in text:
        return "Machine Learning is a subset of AI that allows systems to learn from data and improve automatically."

    elif "what is deep learning" in text:
        return "Deep Learning is a type of machine learning that uses neural networks with multiple layers."


    # Data Science
    elif "what is data science" in text:
        return "Data Science is the process of extracting insights from data using statistics, programming, and machine learning."

    elif "what is nlp" in text:
        return "Natural Language Processing is a field of AI that helps machines understand human language."

    # General Tech
    elif "what is cloud computing" in text:
        return "Cloud computing is the delivery of computing services like storage and servers over the internet."

    elif "what is chatbot" in text:
        return "A chatbot is a program that simulates human conversation using AI and NLP techniques."

    # Default no match found
    return None

In [5]:
# Initial message from chatbot
print("Chatbot: Hello! I am your AI assistant. How can I help you today?")

chat_history_ids = None  # Store conversation history

# Start infinite loop for conversation
while True:
    user_input = input("You: ")   # Take user input

    if user_input.lower() in ["exit", "quit"]:
        print("Chatbot: Goodbye!")
        break

    # Step 1: rule-based
    response = simple_knowledge_response(user_input)

    # Step 2: If no rule matched → use transformer model
    if response is None:
       # Convert user input to tokens
        new_input_ids = tokenizer.encode(user_input + tokenizer.eos_token, return_tensors='pt')

        bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1) if chat_history_ids is not None else new_input_ids

        # Generate response from model
        chat_history_ids = model.generate(
            bot_input_ids,
            max_length=bot_input_ids.shape[-1] + 50,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=False
        )

      # Decode generated tokens into text
        response = tokenizer.decode(
            chat_history_ids[:, bot_input_ids.shape[-1]:][0],
            skip_special_tokens=True
        )

    print("Chatbot:", response)

Chatbot: Hello! I am your AI assistant. How can I help you today?
You: Hii
Chatbot: Hello Sneha! How can I assist you today?
You: What is AI?
Chatbot: Artificial Intelligence is the ability of machines to perform tasks that require human intelligence.
You: What is cloud computing?
Chatbot: Cloud computing is the delivery of computing services like storage and servers over the internet.
You: what is your name?
Chatbot: I am your AI assistant chatbot.
You: exit
Chatbot: Goodbye!
